# Recomendador Basado en Contenido con TF-IDF

Este notebook desarrolla la primera capa semántica real del proyecto.

A diferencia del notebook baseline, aquí ya no se ordenan los POIs solo por rating, score o distancia, sino que se intenta capturar similitud temática entre lugares a partir de su contenido textual.

La finalidad de esta fase es responder a una pregunta clave del TFM:

> dado un POI o una representación textual de un POI, ¿qué otros lugares del dataset son semánticamente parecidos?

Este bloque no genera todavía una ruta final. Su salida debe entenderse como una capa de candidatos relevantes que luego se integrará con señales de ranking, clustering geográfico y lógica de ruta.

## Objetivo del notebook

La metodología seguida en esta fase es incremental:

1. cargar el dataset procesado
2. preparar las variables textuales útiles
3. construir una primera representación (`content_v1`)
4. aplicar TF-IDF y similitud coseno
5. evaluar cualitativamente resultados
6. probar una segunda representación (`content_v2`)
7. comparar ambas versiones y tomar una decisión técnica

El objetivo no es solo “hacer funcionar TF-IDF”, sino decidir qué representación textual resulta más útil y coherente para integrar después en el sistema híbrido.

## 1. Importación de librerías y carga del parquet

En esta sección se cargan las librerías necesarias para el recomendador basado en contenido y se recupera el dataset procesado del proyecto.

In [26]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Cargamos el parquet con los POIs ya procesados previamente
df = pd.read_parquet("../data/pois_barcelona_procesados.parquet")

# Sacamos una muestra visual de los datos para comprobar que se hayan cargado correctamente
df.head()

,id,name,category,subcategory,latitude,longitude,city,description,rating,match_confidence,...,opening_hours_source,has_opening_hours,is_24_7,is_likely_open,has_match_confidence,score,has_valid_source,visit_duration,tags,tags_str
0,3,Biblioteca de Cataluña,cultural,library,41.3810,2.16951,Barcelona,national library,4.2,0.930000,...,OSM,True,False,True,True,3.21900,True,60,"[culture, has_schedule, indoor]",culture|has_schedule|indoor
1,34,Esquerra de l'Eixample,cultural,library,41.3868,2.15205,Barcelona,neighborhood,4.1,0.503766,...,NaN,False,False,False,False,2.56796,False,60,"[culture, indoor]",culture|indoor
2,35,Can Mariner (Barcelona),cultural,library,41.4312,2.16056,Barcelona,masia,4.1,0.650000,...,OSM,True,False,True,True,3.06500,True,60,"[culture, has_schedule, indoor]",culture|has_schedule|indoor
3,111,Archivo Histórico de la Ciudad de Barcelona,cultural,library,41.3841,2.17567,Barcelona,archive,4.3,0.503766,...,NaN,False,False,False,False,2.68696,False,60,"[culture, indoor]",culture|indoor
4,112,Arxiu Diocesà de Barcelona,cultural,library,41.3837,2.17534,Barcelona,episcopal archive,4.2,0.750000,...,OSM,True,False,True,True,3.16500,True,60,"[culture, has_schedule, indoor]",culture|has_schedule|indoor


## 2. Versión 1: TF-IDF + similitud coseno

La primera versión busca una representación textual equilibrada. Para ello se combinan varias fuentes de información en un único campo de contenido.

La lógica de `content_v1` es incluir:

- nombre del POI
- categoría
- subcategoría
- descripción
- etiquetas

Esta representación intenta capturar tanto identidad específica como contexto temático general.

In [27]:
# Creamos un nuevo DataFrame (df_ml) a partir del original (df)
# Seleccionamos solo las columnas relevantes para el modelo de recomendación
df_ml = df[[
    'id',
    'name',
    'category',
    'subcategory',
    'description',
    'tags_str',
    'rating',
    'score',
    'latitude',
    'longitude',
    'visit_duration'
]].copy() # Hacemos una copia para no modificar el DataFrame original

In [28]:
# Rellenamos los valores nulos (NaN) de la columna 'name' con string vacío
# Esto es necesario porque TF-IDF no puede trabajar con valores nulos
df_ml['name'] = df_ml['name'].fillna('')

# Convertimos la columna 'category' a tipo string
# Esto evita problemas porque originalmente es tipo "category" y no se puede usar directamente en texto
df_ml['category'] = df_ml['category'].astype(str)

# Igual que antes, nos aseguramos de que sea texto para poder usarla en TF-IDF
df_ml['subcategory'] = df_ml['subcategory'].astype(str)

# Rellenamos valores nulos en la descripción con string vacío
df_ml['description'] = df_ml['description'].fillna('')

# Rellenamos valores nulos en las etiquetas (tags)
df_ml['tags_str'] = df_ml['tags_str'].fillna('')

### Construcción de `content_v1`

En esta celda se construye el texto que representará cada POI dentro del espacio vectorial TF-IDF. La calidad de este campo es crítica porque condiciona directamente la calidad de las similitudes obtenidas.

In [29]:
# Creamos una nueva columna llamada 'content_v1'
# Esta columna será el texto que representará cada POI para el modelo TF-IDF
df_ml['content_v1'] = (
    # Añadimos el nombre del POI
    df_ml['name'] + ' ' +
    
    # Añadimos la categoría general
    df_ml['category'] + ' ' +
    
    # Añadimos la subcategoría específica
    df_ml['subcategory'] + ' ' +
    
    # Añadimos la descripción
    df_ml['description'] + ' ' +
    
     # Añadimos la descripción
    df_ml['tags_str']
)

### Vectorización y matriz de similitud

Una vez construido `content_v1`, se aplica TF-IDF para transformar cada POI en un vector numérico y después se calcula la similitud coseno entre todos los POIs del dataset.

In [30]:
# Creamos el vectorizador TF-IDF
# max_features=5000 significa que solo se quedará con las 5000 palabras más relevantes
# Esto ayuda a:
# - reducir ruido
# - mejorar rendimiento
# - evitar matrices demasiado grandes
vectorizer_v1 = TfidfVectorizer(max_features=5000)

# Aplicamos el vectorizador al texto ('content_v1')
# fit_transform hace dos cosas:
# 1. Aprende el vocabulario (qué palabras existen y su importancia)
# 2. Convierte cada POI en un vector numérico
df_ml['content_v1'] = df_ml['content_v1'].fillna('').astype(str)
tfidf_matrix_v1 = vectorizer_v1.fit_transform(df_ml['content_v1'])

# Calculamos la similitud coseno entre TODOS los POIs
# Esto genera una matriz donde:
# - cada fila = un POI
# - cada columna = comparación con otro POI
# - cada valor = similitud (entre 0 y 1)
cosine_sim_v1 = cosine_similarity(tfidf_matrix_v1, tfidf_matrix_v1)

In [31]:
# Creamos una Serie de pandas llamada 'indices' y eliminamos los duplicados en los nombres de los POIs, quedandonos si es el caso, con el primero
# Esta Serie nos permite mapear el nombre de un POI → su índice en el DataFrame
indices = pd.Series(df_ml.index, index=df_ml['name']).drop_duplicates()

### Función de recomendación de POIs similares

La siguiente función toma un POI de referencia y devuelve los POIs más similares según la matriz de similitud coseno. La salida de esta función todavía no se interpreta como una ruta, sino como una lista de candidatos semánticamente cercanos.

In [32]:
# Definimos la función que recomienda POIs similares
def recomendar_pois_similares_v1(nombre_poi, top_n=5):
    
    # Comprobamos si el nombre del POI existe en el índice
    # Si no existe, devolvemos un mensaje de error
    if nombre_poi not in indices:
        return f"El POI '{nombre_poi}' no se encuentra en el dataset."
    
    # Obtenemos el índice (posición) del POI dentro del DataFrame
    idx = indices[nombre_poi]
    
    # Obtenemos la lista de similitudes entre este POI y todos los demás
    # enumerate añade el índice de cada elemento → (indice, similitud)
    sim_scores = list(enumerate(cosine_sim_v1[idx]))
    
    # Ordenamos los resultados de mayor a menor similitud
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Eliminamos el primer resultado (es el propio POI, similitud = 1)
    # Nos quedamos con los siguientes top_n más similares
    sim_scores = sim_scores[1:top_n+1]
    
    # Extraemos solo los índices de los POIs recomendados
    poi_indices = [i[0] for i in sim_scores]
    
    # Extraemos las puntuaciones de similitud
    similitudes = [i[1] for i in sim_scores]
    
    # Seleccionamos las filas correspondientes a los POIs recomendados
    # y nos quedamos con las columnas relevantes
    resultados = df_ml.iloc[poi_indices][[
        'id',
        'name',
        'category',
        'subcategory',
        'rating',
        'score',
        'latitude',
        'longitude',
        'visit_duration'
    ]].copy()
    
    # Añadimos la similitud coseno como una nueva columna
    resultados['similaridad_coseno'] = similitudes
    
    # Devolvemos el DataFrame final con las recomendaciones
    return resultados


# Ejecutamos la función con un ejemplo
recomendar_pois_similares_v1("Templo Expiatorio de la Sagrada Familia", top_n=5)

,id,name,category,subcategory,rating,score,latitude,longitude,visit_duration,similaridad_coseno
340,12497,Templo Expiatorio del Sagrado Corazón,religious,church,4.3,2.68696,41.4221,2.11875,40,0.397119
240,6946,Catedral de la Santa Cruz y Santa Eulalia de B...,religious,cathedral,4.4,2.74646,41.3839,2.17624,60,0.371648
262,7302,Escuelas de la Sagrada Familia,cultural,museum,4.3,3.25000,41.4031,2.17425,120,0.340816
434,334753,Iglesia de San Jaime (Barcelona),religious,church,4.8,2.98446,41.3813,2.17550,40,0.257494
328,10990,Basílica de la Merced (Barcelona),religious,church,3.7,2.70700,41.3796,2.17970,40,0.250574


### Revisión de categorías y elección de casos de prueba

Antes de interpretar el modelo, conviene inspeccionar las categorías disponibles y seleccionar algunos POIs representativos para la evaluación cualitativa.

In [33]:
# Vemos que categorias hay para realizar las pruebas de recomendacion
df_ml["category"].value_counts()

category
cultural              331
monument              100
tourist_attraction     84
fountain               75
park                   61
religious              44
historic               41
administrative         35
bridge                  3
unknown                 1
Name: count, dtype: int64

In [34]:
# En esta celda sacamos una muestra visual de algunas categorias para ver que POIs hay y realizar las pruebas después
display(df_ml[df_ml["category"] == "park"].head())
display(df_ml[df_ml["category"] == "monument"].head())
display(df_ml[df_ml["category"] == "historic"].head())

,id,name,category,subcategory,description,tags_str,rating,score,latitude,longitude,visit_duration,content_v1
26,491,Parque Güell,park,conservatory,sculpture garden,culture|indoor,4.8,2.98446,41.4136,2.15294,60,Parque Güell park conservatory sculpture garde...
48,842,Jardines del Umbráculo,park,conservatory,garden,culture|indoor,4.8,2.98446,41.3687,2.15770,60,Jardines del Umbráculo park conservatory garde...
161,5562,Jardines Mossèn Costa i Llobera,park,conservatory,botanical garden,culture|has_schedule|indoor,4.7,3.41000,41.3689,2.17213,60,Jardines Mossèn Costa i Llobera park conservat...
220,6480,Parque del Laberinto de Horta,park,conservatory,garden,culture|indoor,4.6,2.86546,41.4403,2.14556,60,Parque del Laberinto de Horta park conservator...
421,19518,Jardines de Joan Brossa,park,conservatory,building,culture|indoor,4.8,2.98446,41.3684,2.16702,60,Jardines de Joan Brossa park conservatory buil...


,id,name,category,subcategory,description,tags_str,rating,score,latitude,longitude,visit_duration,content_v1
50,875,Bombardeos aéreos de Barcelona en enero de 1938,monument,memorial,battle,has_schedule|historic|quick_visit|touristic,4.4,3.26600,41.3833,2.17505,20,Bombardeos aéreos de Barcelona en enero de 193...
75,2040,Barcelona a Prim,monument,memorial,sculpture,historic|quick_visit|touristic,4.4,2.74646,41.3864,2.18680,20,Barcelona a Prim monument memorial sculpture h...
76,2097,Buenaventura Durruti,monument,memorial,human,historic|quick_visit|touristic,3.5,2.21096,41.3540,2.15013,20,Buenaventura Durruti monument memorial human h...
77,2130,A los mártires de la independencia,monument,memorial,sculpture,has_schedule|historic|quick_visit|touristic,4.5,3.29400,41.3836,2.17578,20,A los mártires de la independencia monument me...
78,2143,Cinema Comèdia,monument,memorial,movie theater,historic|quick_visit|touristic,4.3,2.68696,41.3881,2.16631,20,Cinema Comèdia monument memorial movie theater...


,id,name,category,subcategory,description,tags_str,rating,score,latitude,longitude,visit_duration,content_v1
35,749,Casa de l'Ardiaca,historic,castle,building,has_schedule|historic|long_visit|touristic,4.9,3.55600,41.3841,2.17567,90,Casa de l'Ardiaca historic castle building has...
54,1017,Turó de la Rovira,historic,bunker,mountain,quick_visit,4.3,2.68696,41.4194,2.16174,20,Turó de la Rovira historic bunker mountain qui...
56,1171,Monasterio de San Jerónimo del Valle de Hebrón,historic,ruins,destroyed monastery,historic|touristic,4.4,2.74646,41.4305,2.12791,45,Monasterio de San Jerónimo del Valle de Hebrón...
58,1174,Can Calopa de Dalt,historic,ruins,building,historic|touristic,4.2,2.62746,41.4291,2.06170,45,Can Calopa de Dalt historic ruins building his...
59,1175,Torre de Santa Margarida,historic,ruins,masia,historic|touristic,4.9,3.04396,41.4067,2.07128,45,Torre de Santa Margarida historic ruins masia ...


### Evaluación cualitativa de la versión 1

La evaluación en esta fase es cualitativa y exploratoria. Se utilizan varios POIs conocidos para comprobar si las recomendaciones mantienen coherencia temática, categoría similar y una relación semántica razonable.

In [35]:
# Lista de POIs de prueba
# Estos son ejemplos representativos que usamos para evaluar el modelo
pois_prueba = [
    "Templo Expiatorio de la Sagrada Familia",
    "Parque Güell",                             
    "Barcelona a Prim",                         
    "Casa de l'Ardiaca",                        
]

# Recorremos cada POI de prueba
for poi in pois_prueba:
    
    # Imprimimos separadores para que la salida sea más legible
    print(f"\n{'='*60}")
    
    # Mostramos qué POI estamos evaluando
    print(f"POI consultado: {poi}")
    
    print(f"{'='*60}")
    
    # Llamamos a la función de recomendación
    resultado = recomendar_pois_similares_v1(poi, top_n=5)
    
    # Si la función devuelve un string (error porque no encuentra el POI)
    if isinstance(resultado, str):
        print(resultado)
    
    else:
        # Mostramos los resultados en formato tabla (más visual que print)
        display(resultado[[
            'name',                  # nombre del POI recomendado
            'category',              # categoría
            'subcategory',           # subcategoría
            'rating',                # valoración
            'similaridad_coseno',     # qué tan parecido es al POI original
            'latitude',              # latitud (para ubicación)
            'longitude',             # longitud (para ubicación)
            'visit_duration'         # duración recomendada de la visita
        ]])


POI consultado: Templo Expiatorio de la Sagrada Familia


,name,category,subcategory,rating,similaridad_coseno,latitude,longitude,visit_duration
340,Templo Expiatorio del Sagrado Corazón,religious,church,4.3,0.397119,41.4221,2.11875,40
240,Catedral de la Santa Cruz y Santa Eulalia de B...,religious,cathedral,4.4,0.371648,41.3839,2.17624,60
262,Escuelas de la Sagrada Familia,cultural,museum,4.3,0.340816,41.4031,2.17425,120
434,Iglesia de San Jaime (Barcelona),religious,church,4.8,0.257494,41.3813,2.17550,40
328,Basílica de la Merced (Barcelona),religious,church,3.7,0.250574,41.3796,2.17970,40



POI consultado: Parque Güell


,name,category,subcategory,rating,similaridad_coseno,latitude,longitude,visit_duration
220,Parque del Laberinto de Horta,park,conservatory,4.6,0.529707,41.4403,2.14556,60
423,Parque de Cervantes,park,conservatory,4.7,0.489379,41.3844,2.10575,60
644,Parque de la Ciudadela,park,conservatory,4.4,0.470702,41.3881,2.18752,60
550,Parque del Guinardó,park,conservatory,4.4,0.461877,41.4195,2.16961,60
542,Parque de la Trinidad,park,conservatory,4.5,0.446218,41.4493,2.19565,60



POI consultado: Barcelona a Prim


,name,category,subcategory,rating,similaridad_coseno,latitude,longitude,visit_duration
467,A Capmany,monument,memorial,4.7,0.443241,41.3760,2.17606,20
83,A Rafael Casanova,monument,memorial,4.6,0.429737,41.3909,2.17752,20
717,La ben plantada,monument,memorial,4.8,0.418551,41.3946,2.14010,20
79,A Joan Güell i Ferrer,monument,memorial,3.9,0.413775,41.3887,2.16729,20
700,Vigilant 2,monument,monument,4.5,0.401100,41.4407,2.14462,25



POI consultado: Casa de l'Ardiaca


,name,category,subcategory,rating,similaridad_coseno,latitude,longitude,visit_duration
309,Casa Padellás,historic,castle,4.7,0.597828,41.3840,2.17776,90
215,Casa Vicens,historic,castle,3.6,0.584608,41.4035,2.15063,90
211,Castillo de Montjuic,historic,castle,4.5,0.534349,41.3634,2.16599,90
287,Palacio Robert,historic,castle,4.5,0.521854,41.3961,2.15941,90
310,Palau Centelles,historic,castle,4.7,0.516065,41.3816,2.17705,90


## 3. Versión 2: TF-IDF + similitud coseno con más peso temático

La segunda versión intenta reforzar la dimensión temática del contenido dando más peso a subcategorías y etiquetas. La hipótesis es que esto podría mejorar algunos casos concretos, aunque también existe el riesgo de volver excesivamente homogéneas las recomendaciones.

In [36]:
# Creamos una nueva columna llamada 'content_v2'
# Esta es una segunda versión del texto que representa cada POI
# El objetivo es probar si cambiando la importancia de las variables mejora la recomendación
df_ml['content_v2'] = (
    df_ml['category'] + ' ' +
    df_ml['subcategory'] + ' ' +
    
    # Repetimos la subcategoría para darle más peso en el modelo
    df_ml['subcategory'] + ' ' +
    df_ml['tags_str'] + ' ' +
    df_ml['tags_str'] + ' ' +
    df_ml['description']
)

In [37]:
# Creamos el vectorizador TF-IDF para la versión 2 del modelo
vectorizer_v2 = TfidfVectorizer(
    
    # max_features=5000 limita el vocabulario a las 5000 palabras/expresiones más relevantes
    # Esto ayuda a reducir ruido, memoria y tiempo de cálculo
    max_features=5000,
    
    # ngram_range=(1, 2) indica que el modelo usará:
    # - unigramas: palabras sueltas
    # - bigramas: combinaciones de dos palabras seguidas
    # Esto permite capturar algo más de contexto semántico
    ngram_range=(1, 2)
)

# Ajustamos el vectorizador al texto de 'content_v2' y transformamos cada POI en un vector numérico
# fit_transform hace dos cosas:
# 1. Aprende el vocabulario del dataset
# 2. Convierte cada POI en una representación TF-IDF
df_ml['content_v2'] = df_ml['content_v2'].fillna('').astype(str)
tfidf_matrix_v2 = vectorizer_v2.fit_transform(df_ml['content_v2'])

# Calculamos la similitud coseno entre todos los POIs de la versión 2
cosine_sim_v2 = cosine_similarity(tfidf_matrix_v2, tfidf_matrix_v2)

In [38]:
# Definimos la función que recomienda POIs similares usando la versión 2 del modelo
def recomendar_pois_similares_v2(nombre_poi, top_n=5):
    
    # Comprobamos si el nombre del POI existe en el índice
    # Si no existe, devolvemos un mensaje para evitar errores
    if nombre_poi not in indices:
        return f"El POI '{nombre_poi}' no se encuentra en el dataset."
    
    # Obtenemos el índice (posición) del POI en el DataFrame
    idx = indices[nombre_poi]
    
    # Recuperamos las similitudes entre el POI consultado y todos los demás POIs
    # enumerate convierte el vector de similitudes en pares (indice, similitud)
    sim_scores = list(enumerate(cosine_sim_v2[idx]))
    
    # Ordenamos los resultados de mayor a menor similitud
    # Así los POIs más parecidos quedan al principio
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Eliminamos el primer elemento, que siempre es el propio POI
    # (porque su similitud consigo mismo es 1)
    # Después nos quedamos con los siguientes top_n resultados
    sim_scores = sim_scores[1:top_n+1]
    
    # Extraemos solo los índices de los POIs recomendados
    poi_indices = [i[0] for i in sim_scores]
    
    # Extraemos solo los valores de similitud coseno
    similitudes = [i[1] for i in sim_scores]
    
    # Seleccionamos las filas del DataFrame que corresponden a los POIs recomendados
    # y nos quedamos con las columnas relevantes para análisis posterior
    resultados = df_ml.iloc[poi_indices][[
        'id',
        'name',
        'category',
        'subcategory',
        'rating',
        'score',
        'latitude',
        'longitude',
        'visit_duration'
    ]].copy()
    
    # Añadimos una nueva columna con la similitud coseno calculada
    resultados['similaridad_coseno'] = similitudes
    
    # Devolvemos el DataFrame final con las recomendaciones
    return resultados

### Comparación cualitativa entre V1 y V2

En esta etapa se comparan ambas versiones usando los mismos POIs de prueba. El objetivo no es buscar una métrica automática compleja, sino evaluar si los resultados siguen siendo coherentes, variados y útiles para una integración posterior.

In [39]:
# Recorremos cada POI de prueba
for poi in pois_prueba:
    
    # Imprimimos una línea separadora más visual (con # en vez de =)
    print(f"\n{'#'*70}")
    
    # Mostramos el POI que estamos evaluando
    print(f"POI consultado: {poi}")
    
    print(f"{'#'*70}")
    
    # ---------------------------
    # RESULTADOS DEL MODELO V1
    # ---------------------------
    print("\n--- Resultados V1 ---")
    
    # Ejecutamos el recomendador basado en content_v1
    resultado_v1 = recomendar_pois_similares_v1(poi, top_n=5)
    
    # Si devuelve un string (error porque no encuentra el POI)
    if isinstance(resultado_v1, str):
        print(resultado_v1)
    
    else:
        # Mostramos los resultados en formato tabla
        # Solo mostramos columnas relevantes para comparar
        display(resultado_v1[[
             'name',                  # nombre del POI recomendado
            'category',              # categoría
            'subcategory',           # subcategoría
            'rating',                # valoración
            'similaridad_coseno',     # qué tan parecido es al POI original
            'latitude',              # latitud (para ubicación)
            'longitude',             # longitud (para ubicación)
            'visit_duration'         # duración recomendada de la visita
        ]])
    
    # ---------------------------
    # RESULTADOS DEL MODELO V2
    # ---------------------------
    print("\n--- Resultados V2 ---")
    
    # Ejecutamos el recomendador basado en content_v2
    resultado_v2 = recomendar_pois_similares_v2(poi, top_n=5)
    
    # Mismo control de errores que en V1
    if isinstance(resultado_v2, str):
        print(resultado_v2)
    
    else:
        # Mostramos los resultados de la V2
        display(resultado_v2[[
            'name',                  # nombre del POI recomendado
            'category',              # categoría
            'subcategory',           # subcategoría
            'rating',                # valoración
            'similaridad_coseno',     # qué tan parecido es al POI original
            'latitude',              # latitud (para ubicación)
            'longitude',             # longitud (para ubicación)
            'visit_duration'         # duración recomendada de la visita
        ]])


######################################################################
POI consultado: Templo Expiatorio de la Sagrada Familia
######################################################################

--- Resultados V1 ---


,name,category,subcategory,rating,similaridad_coseno,latitude,longitude,visit_duration
340,Templo Expiatorio del Sagrado Corazón,religious,church,4.3,0.397119,41.4221,2.11875,40
240,Catedral de la Santa Cruz y Santa Eulalia de B...,religious,cathedral,4.4,0.371648,41.3839,2.17624,60
262,Escuelas de la Sagrada Familia,cultural,museum,4.3,0.340816,41.4031,2.17425,120
434,Iglesia de San Jaime (Barcelona),religious,church,4.8,0.257494,41.3813,2.17550,40
328,Basílica de la Merced (Barcelona),religious,church,3.7,0.250574,41.3796,2.17970,40



--- Resultados V2 ---


,name,category,subcategory,rating,similaridad_coseno,latitude,longitude,visit_duration
240,Catedral de la Santa Cruz y Santa Eulalia de B...,religious,cathedral,4.4,0.781510,41.3839,2.17624,60
434,Iglesia de San Jaime (Barcelona),religious,church,4.8,0.464863,41.3813,2.17550,40
339,Monasterio de San Pablo del Campo,religious,church,4.8,0.424977,41.3763,2.16952,40
316,Parroquia de San Juan Bautista de Gracia,religious,church,4.7,0.388770,41.4052,2.15691,40
317,Església de Sant Andreu de Palomar,religious,church,4.6,0.388770,41.4364,2.19182,40



######################################################################
POI consultado: Parque Güell
######################################################################

--- Resultados V1 ---


...

,name,category,subcategory,rating,similaridad_coseno,latitude,longitude,visit_duration
83,A Rafael Casanova,monument,memorial,4.6,1.000000,41.3909,2.17752,20
86,El Beso de la Muerte,monument,memorial,4.3,1.000000,41.3962,2.20420,20
717,La ben plantada,monument,memorial,4.8,1.000000,41.3946,2.14010,20
467,A Capmany,monument,memorial,4.7,0.966674,41.3760,2.17606,20
569,Al Doctor Hahnemann,monument,memorial,4.5,0.966674,41.3920,2.13720,20



######################################################################
POI consultado: Casa de l'Ardiaca
######################################################################

--- Resultados V1 ---


,name,category,subcategory,rating,similaridad_coseno,latitude,longitude,visit_duration
309,Casa Padellás,historic,castle,4.7,0.597828,41.3840,2.17776,90
215,Casa Vicens,historic,castle,3.6,0.584608,41.4035,2.15063,90
211,Castillo de Montjuic,historic,castle,4.5,0.534349,41.3634,2.16599,90
287,Palacio Robert,historic,castle,4.5,0.521854,41.3961,2.15941,90
310,Palau Centelles,historic,castle,4.7,0.516065,41.3816,2.17705,90



--- Resultados V2 ---


,name,category,subcategory,rating,similaridad_coseno,latitude,longitude,visit_duration
287,Palacio Robert,historic,castle,4.5,1.000000,41.3961,2.15941,90
305,Palau de les Heures,historic,castle,4.6,1.000000,41.4368,2.14142,90
310,Palau Centelles,historic,castle,4.7,1.000000,41.3816,2.17705,90
215,Casa Vicens,historic,castle,3.6,0.932561,41.4035,2.15063,90
211,Castillo de Montjuic,historic,castle,4.5,0.928486,41.3634,2.16599,90


## Interpretación de resultados

La comparación entre ambas versiones debe leerse en clave metodológica:

- una similitud muy alta no implica necesariamente una recomendación mejor
- el objetivo no es solo agrupar POIs parecidos, sino mantener capacidad de diferenciación
- un exceso de peso en subcategorías o tags puede volver demasiado rígido el modelo

Por eso, en esta fase es preferible una representación equilibrada antes que una especialización excesiva.

# ***Conclusiones del notebook:***
-   Se ha implementado un recomendador basado en contenido usando TF-IDF y similitud coseno.
-   La versión V1 ofrece resultados más equilibrados y variados.
-   La versión V2, al dar más peso a subcategory y tags, mejora algunos casos concretos pero también genera similitudes excesivamente altas entre POIs distintos.
-   Por ello, se toma V1 como baseline del recomendador por contenido para integrarlo posteriormente con los siguientes módulos del sistema híbrido.

---

# ***Decisión final de esta fase:***
-   Se selecciona content_v1 como versión base del recomendador por contenido, ya que ofrece recomendaciones más coherentes y mejor capacidad de diferenciación entre POIs que content_v2.

La salida de este recomendador no se interpreta todavía como una ruta final, sino como un conjunto de POIs candidatos semánticamente similares.

Estos resultados se combinarán en fases posteriores con:
-   Clustering geográfico
-   Ranking adicional
-   Algoritmo de optimización de ruta